# Exploratory data analysis (EDA)

In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from zurich_uhi_lur import plot_utils

figwidth, figheight = plt.rcParams["figure.figsize"]
sns.set_style("whitegrid")

In [ ]:
# spatial extent
# agglom_extent_filepath = "../data/raw/zurich-extent.gpkg"
agglom_extent_filepath = "../data/processed/zurich-lcd-extent.gpkg"
stations_gdf_filepath = "../data/processed/stations.gpkg"
ts_df_filepath = "../data/processed/ts-df.csv"

t_label = "$T \;$ [$^{\circ}$C]"
t_mean_label = "$T_{mean} \;$ [$^{\circ}$C]"
t_minus_mean_label = "$T - T_{mean} \;$ [$^{\circ}$C]"
t_std_label = "$T_{std} \;$ [$^{\circ}$C]"

dst_t_ts_filepath = "../reports/figures/t-ts.pdf"
dst_t_mean_maps_filepath = "../reports/figures/station-t-mean-maps.png"
dst_t_minus_mean_maps_filepath = "../reports/figures/station-t-t-mean-maps.png"

# for plots
compare_maps_kwargs = dict(facecolor=(0.87, 0.56, 0.02, 0.2), edgecolor="gray")
savefig_map_kwargs = dict(bbox_inches="tight", dpi=300)

In [ ]:
region = gpd.read_file(agglom_extent_filepath)["geometry"].iloc[:1]

In [ ]:
stations_gdf = gpd.read_file(stations_gdf_filepath)
# get the station id column label
station_id_col = stations_gdf.columns.difference(["geometry", "source"])[0]
# set the station id as index
stations_gdf = stations_gdf.set_index(station_id_col)
# show the data frame
stations_gdf

In [ ]:
ts_df = pd.read_csv(
    ts_df_filepath, index_col=["heatwave", "time"], parse_dates=["time"]
)
ts_df_dict = {
    source: ts_df[source_ser.index]
    for source, source_ser in stations_gdf.groupby("source", sort=False)
}

In [ ]:
# num_cols = len(ts_df_dict)

# fig, axes = plt.subplots(
#     1, num_cols, figsize=(num_cols * figwidth, figheight), sharey=True
# )
# for i, (source, source_ts_df) in enumerate(ts_df_dict.items()):
#     plot_ts_df = source_ts_df.reset_index().melt(
#         id_vars=["heatwave", "time"], var_name="station_id", value_name="T"
#     )
#     plot_ts_df["hour"] = plot_ts_df["time"].dt.hour
#     sns.lineplot(data=plot_ts_df, x="hour", y="T", hue="heatwave", ax=axes[i])
#     # plot_utils.hourly_lineplot(
#     #     source_ts_df.rename(columns={source_ts_df.columns[0]: t_label}),
#     #     ax=axes[i],
#     #     legend=i == 0,
#     # )
#     axes[i].set_title(source)
#     axes[i].set_xlim([0, 23])
plot_ts_df = pd.concat(
    [
        source_ts_df.reset_index()
        .melt(id_vars=["heatwave", "time"], var_name="station_id", value_name="T")
        .assign(source=source)
        for _, (source, source_ts_df) in enumerate(ts_df_dict.items())
    ],
    ignore_index=True,
)
plot_ts_df["hour"] = plot_ts_df["time"].dt.hour
ax = sns.lineplot(data=plot_ts_df, x="hour", y="T", hue="source")
ax.set_xlim([0, 23])
ax.figure.tight_layout()
ax.figure.savefig(dst_t_ts_filepath)

In [ ]:
fig = plot_utils.compare_maps(
    stations_gdf,
    pd.concat(
        [source_ts_df.mean() for _, source_ts_df in ts_df_dict.items()],
        axis="rows",
    ),
    var_label=t_mean_label,
    region_gdf=region,
    **compare_maps_kwargs,
)
fig.savefig(dst_t_mean_maps_filepath, **savefig_map_kwargs)

In [ ]:
t_minus_mean_ser = pd.concat(
    [
        source_ts_df.apply(lambda row: row - row.mean(), axis="columns").mean()
        for _, source_ts_df in ts_df_dict.items()
    ],
    axis="rows",
)
max_abs = t_minus_mean_ser.abs().max()
fig = plot_utils.compare_maps(
    stations_gdf,
    t_minus_mean_ser,
    var_label=t_minus_mean_label,
    region_gdf=region,
    vmin=-max_abs,
    vmax=max_abs,
    **compare_maps_kwargs,
)
fig.savefig(dst_t_minus_mean_maps_filepath, **savefig_map_kwargs)

In [ ]:
fig = plot_utils.compare_maps(
    stations_gdf,
    pd.concat(
        [
            source_ts_df.apply(lambda row: row - row.mean(), axis="columns").std()
            for _, source_ts_df in ts_df_dict.items()
        ],
        axis="rows",
    ),
    var_label=t_std_label,
    region_gdf=region,
    **compare_maps_kwargs,
)